In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from spikingjelly.activation_based import neuron, surrogate, functional, layer

from src.cams.gradcam import SNNGradCAM
from src.cams.ablation_cam import AblationCAM
from src.cams.mc_ablation_cam import MCAblationCAM
from src.cams.sam import SpikeActivationMap
from src.cams.guided_backprop import GuidedBackpropCAM

from src.utils.config_utils import setup_config_dictionary
from src.utils.data_utils import get_dataloaders
from src.utils.model_utils import get_model
from src.utils.save_utils import interactive_load_model, load_model
from src.utils.plot_utils import plot_frames, plot_cam_time_series, combine_polarities, plot_aggregated_cam

# Repro + device
print(torch.seed())
# torch.manual_seed(1937388963656961197)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = setup_config_dictionary()
config["DATASET_NAME"] = "dvs_gesture"
config["MODEL_NAME"] = "simple_snn_no_vote"
config["TAU"] = 2.0
config["N_TIME_BINS"] = 16
# the number of time steps must be set from the config file, since the dataloader reads it from there

print('Device:', device)
print('Dataset:', config['DATASET_NAME'])
print('Model:', config['MODEL_NAME'])

In [ ]:
train_loader, test_loader = get_dataloaders(
    config['DATASET_NAME'],
    batch_size=1,
    n_time_bins=config["N_TIME_BINS"],
)

sample_events, sample_labels = next(iter(train_loader))
# encoder = PoissonEncoder(step_mode='m')
encoder = None
was_single_image = False

if sample_events.dim() == 4:
    was_single_image = True
    sample_events = sample_events.unsqueeze(1)  # add time dimension
    sample_events = sample_events.repeat(1, config["N_TIME_BINS"], 1, 1, 1)  # repeat along time dimension

if encoder is not None:
    sample_events = encoder(sample_events)

sample_events = sample_events.transpose(0, 1)

T, B, C, H, W = sample_events.shape
print('Events Shape (T, B, C, H, W):', (T, B, C, H, W))

num_classes = len(train_loader.dataset.classes) if hasattr(train_loader.dataset, "classes") else len(train_loader.dataset.dataset.classes)
spiking_neuron = neuron.LIFNode if not was_single_image else neuron.IFNode

print("The spiking neuron used is:", spiking_neuron)

params = {
    "model_name": config["MODEL_NAME"],
    "num_classes": num_classes,
    "in_channels": C,
    "spiking_neuron": spiking_neuron,
    "pretrained": False,
    "detach_reset": False,
    "surrogate_function": surrogate.ATan(),
}
if not was_single_image:
    params["tau"] = config["TAU"]
model = get_model(**params)

functional.set_step_mode(model, 'm')
functional.set_backend(model, "torch", instance=spiking_neuron)
model.to(device)

# uncomment to pick which model to load, otherwise just run load_model, which loads the latest
# interactive_load_model(model, config['MODEL_SAVE_DIR'], config['DATASET_NAME'])
load_model(model, config["MODEL_SAVE_DIR"], config["DATASET_NAME"])

model.eval()
print("Model loaded.")

In [ ]:
def cam_variance_T_HW(cam):
    # cam: (T, H, W). Returns mean over time of per-frame spatial variance.
    import torch
    if isinstance(cam, torch.Tensor):
        cam = cam.detach().cpu().numpy()
    return float(cam.var(axis=(1, 2)).mean())

def build_model_copy():
    params = {
        "model_name": config["MODEL_NAME"],
        "num_classes": num_classes,
        "in_channels": C,
        "spiking_neuron": spiking_neuron,
        "pretrained": False,
        "detach_reset": False,
        "surrogate_function": surrogate.ATan(),
    }
    if not was_single_image:
        params["tau"] = config["TAU"]
    new_model = get_model(**params)
    functional.set_step_mode(new_model, 'm')
    functional.set_backend(new_model, "torch", instance=spiking_neuron)
    new_model.load_state_dict(model.state_dict(), strict=True)
    new_model.to(device)
    new_model.eval()
    return new_model

In [ ]:
conv_layers = [m for m in model.modules() if isinstance(m, layer.Conv2d)]
lif_layers = [m for m in model.modules() if isinstance(m, spiking_neuron)]
print(f'Found {len(conv_layers)} conv layers and {len(lif_layers)} (L)IF layers.')

for events, labels in test_loader:
    was_single_image = False
    if events.dim() == 4:
        was_single_image = True
        events = events.unsqueeze(1)  # add time dimension
        events = events.repeat(1, config["N_TIME_BINS"], 1, 1, 1)  # repeat along time dimension
    
    pre_encoding_events = events.clone()
    if encoder is not None:
        events = encoder(events)

    events = events.transpose(0, 1).to(device)
    label = labels.to(device)
    print('Label:', int(label[0].item()))

    outputs = model(events)  # (T, B, num_classes)
    time_sum = outputs.sum(dim=0)
    # print the outputs per class if there are less than 20 classes
    if num_classes <= 20:
        print(f"Outputs per class over time: {time_sum[0].detach().cpu().numpy()}")
    pred = time_sum.argmax(dim=1)
    print(f'Predicted class: {int(pred[0].item())} with confidence {F.softmax(time_sum, dim=1)[0,pred].item():.4f}')
    
    if not pred[0].item() == label[0].item():
        print("Skipping CAM generation since prediction is incorrect.")
        continue

    frames = pre_encoding_events[0, :].detach().cpu()  # (T, C, H, W)
    if frames.shape[1] == 2:  # (T, 2, H, W) -> (T, 1, H, W)
        combined = combine_polarities(frames.permute(0,2,3,1).numpy())  # (T,H,W)
        combined = np.expand_dims(combined, axis=1)
    else: # 3 channels, assume RGB
        combined = frames.numpy()

    plot_frames(combined)
    mc_var_by_layer = []
    naive_var_by_layer = []
    sam_var_by_layer = []
    grad_var_by_layer = []
    for layer_index in range(len(conv_layers)):
        print(f'Targeting layer index {layer_index} for CAMs, out of {len(conv_layers)} conv layers and {len(lif_layers)} LIF layers.')
        # Grad-CAM
        gc_model = build_model_copy()
        for p in gc_model.parameters():
            p.requires_grad_(True)
        functional.reset_net(gc_model)
        gc_conv_layers = [m for m in gc_model.modules() if isinstance(m, layer.Conv2d)]
        gc_target_layer = gc_conv_layers[layer_index]
        grad_cam = SNNGradCAM(gc_model, gc_target_layer)
        gc_model.train()
        for m in gc_model.modules():
            if isinstance(m, (layer.BatchNorm1d, layer.BatchNorm2d, layer.BatchNorm3d, layer.Dropout, layer.Dropout2d)):
                m.eval()
        grad_cams = grad_cam.generate(events, class_idx=pred[0].item())
        print("Grad-CAMs over time:")
        if was_single_image: plot_aggregated_cam(combined, grad_cams)
        print(f"combined shape: {combined.shape}, grad_cams shape: {grad_cams.shape}")
        plot_cam_time_series(combined, grad_cams, per_frame_cam_norm=False, images_per_row=8)

        # Monte Carlo Ablation CAM
        mc_model = build_model_copy()
        functional.reset_net(mc_model)
        mc_lif_layers = [m for m in mc_model.modules() if isinstance(m, spiking_neuron)]
        mc_target_layer = mc_lif_layers[layer_index]
        k_trials = 50
        mc_ablation_cam = MCAblationCAM(mc_model, mc_target_layer, channel_batch_size=16, k_trials=k_trials)
        mc_ablation_cams = mc_ablation_cam.generate(events, class_idx=pred[0].item())
        print("MC Ablation CAMs over time:")
        if was_single_image: plot_aggregated_cam(combined, mc_ablation_cams)
        plot_cam_time_series(combined, mc_ablation_cams, per_frame_cam_norm=False, images_per_row=8)
        
        # Ablation CAM
        na_cam = build_model_copy()
        functional.reset_net(na_cam)
        na_conv_layers = [m for m in na_cam.modules() if isinstance(m, layer.Conv2d)]
        na_target_layer = na_conv_layers[layer_index]
        ab_cam = AblationCAM(na_cam, na_target_layer, channel_batch_size=16)
        ab_cams = ab_cam.generate(events, class_idx=pred[0].item())
        print("Ablation CAMs over time:")
        if was_single_image: plot_aggregated_cam(combined, ab_cams)
        plot_cam_time_series(combined, ab_cams, per_frame_cam_norm=False, images_per_row=8)

        # Spike Activation Map
        sam_model = build_model_copy()
        functional.reset_net(sam_model)
        sam_lif_layers = [m for m in sam_model.modules() if isinstance(m, spiking_neuron)]
        sam_target_layer = sam_lif_layers[layer_index]
        spike_activation_map = SpikeActivationMap(sam_model, sam_target_layer)
        sams = spike_activation_map.generate(events)
        print("SAMs over time:")
        if was_single_image: plot_aggregated_cam(combined, sams)
        plot_cam_time_series(combined, sams, per_frame_cam_norm=False, images_per_row=8)

        # Guided Backprop CAM
        gb_model = build_model_copy()
        for p in gb_model.parameters():
            p.requires_grad_(True)
        functional.reset_net(gb_model)
        guided_backprop_cam = GuidedBackpropCAM(gb_model)
        gb_model.train()
        for m in gb_model.modules():
            if isinstance(m, (layer.BatchNorm1d, layer.BatchNorm2d, layer.BatchNorm3d, layer.Dropout, layer.Dropout2d)):
                m.eval()
        gb_cams = guided_backprop_cam.generate(events, class_idx=pred[0].item(), upsample=False)
        print("Guided Backprop CAMs over time:")
        if was_single_image: plot_aggregated_cam(combined, gb_cams)
        plot_cam_time_series(combined, gb_cams, per_frame_cam_norm=False, images_per_row=8)

        baseline_conv = mc_ablation_cam.convergence_data['baseline']
        x_axis = range(1, k_trials + 1)
        ch_10_conv = mc_ablation_cam.convergence_data['ch_10']
        plt.plot(x_axis, baseline_conv, label="Baseline Score")
        plt.plot(x_axis, ch_10_conv, label="Channel 10 Ablated Score")
        plt.xlabel("Number of Trials (k)")
        plt.ylabel("Average Score")
        plt.legend()
        plt.title("MC Ablation CAM Score Convergence")
        plt.show()

        mc_var_by_layer.append(cam_variance_T_HW(mc_ablation_cams))
        naive_var_by_layer.append(cam_variance_T_HW(ab_cams))
        sam_var_by_layer.append(cam_variance_T_HW(sams))
        grad_var_by_layer.append(cam_variance_T_HW(grad_cams))
        print("Done processing for this layer.")

    plt.figure(figsize=(8, 4))
    xs = list(range(len(grad_var_by_layer)))
    plt.plot(xs, mc_var_by_layer, marker="o", label="MC Ablation CAM")
    plt.plot(xs, naive_var_by_layer, marker="o", label="Ablation CAM")
    plt.plot(xs, sam_var_by_layer, marker="o", label="SAM")
    plt.plot(xs, grad_var_by_layer, marker="o", label="Grad-CAM")
    plt.xlabel("Layer index")
    plt.ylabel("Mean pixel variance (avg over T)")
    plt.title("CAM spatial variance across layers")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    break  # only do one sample

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from collections import defaultdict
from tqdm.auto import tqdm
from contextlib import contextmanager

@contextmanager
def managed_model_copy():
    """Creates a model copy and ensures it is deleted after use to free VRAM."""
    model = build_model_copy()
    try:
        yield model
    finally:
        del model

# Set style for plots
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 12})

# --- Configuration ---
# Define pairs of (Conv Layer Index, Spiking Layer Index) to target.
# Adjust these indices to match corresponding blocks in your model.
# The user requested indices 6, 9, 12. We will validate these against the model depth.
requested_layer_pairs = [
    (0, 0),   
    (1, 1),
    (2, 2),
    (3, 3),   
    (4, 4)
]
# Fallback for smaller models (like simple_snn) if the above are out of bounds
# requested_layer_pairs = [(0, 0), (1, 1)] 

top_percent = 0.2
# TODO: Increase this for actual reporting
max_samples = 100  # Set to None for full test set

methods = ['grad', 'ablation', 'mc_ablation', 'sam', 'random']

# --- Helpers ---

def get_conv_layers(m):
    return [x for x in m.modules() if isinstance(x, layer.Conv2d)]

def get_lif_layers(m):
    # Detects both LIF and IF nodes
    return [x for x in m.modules() if isinstance(x, (neuron.LIFNode, neuron.IFNode))]

def get_valid_pairs(m, pairs):
    n_conv = len(get_conv_layers(m))
    n_lif = len(get_lif_layers(m))
    valid = []
    for c, l in pairs:
        if c < n_conv and l < n_lif:
            valid.append((c, l))
    print(f"Validating layers: Requested {pairs}, Keeping {valid} (Model has {n_conv} Conv, {n_lif} LIF)")
    return valid

def generate_grad_cam(events, class_idx, target_layer_idx, neuron_type):
    with managed_model_copy() as gc_model:
        # Workaround: Enable gradients for backprop
        for p in gc_model.parameters():
            p.requires_grad_(True)
        functional.reset_net(gc_model)
        
        gc_convs = get_conv_layers(gc_model)
        target = gc_convs[target_layer_idx]
        
        grad_cam = SNNGradCAM(gc_model, target)
        
        # Train mode for gradients, but eval for BN/Dropout
        gc_model.train()
        for m in gc_model.modules():
            if isinstance(m, (layer.BatchNorm1d, layer.BatchNorm2d, layer.BatchNorm3d, layer.Dropout, layer.Dropout2d)):
                m.eval()
                
        return grad_cam.generate(events, class_idx=class_idx).detach()

def generate_ablation_cam(events, class_idx, target_layer_idx, neuron_type):
    with managed_model_copy() as ab_model:
        target = get_conv_layers(ab_model)[target_layer_idx]
        ab_cam = AblationCAM(ab_model, target, channel_batch_size=16)
        return ab_cam.generate(events, class_idx=class_idx).detach()

def generate_mc_ablation_cam(events, class_idx, target_layer_idx, neuron_type):
    with managed_model_copy() as mc_ablation_model:
        functional.reset_net(mc_ablation_model)
        target = get_lif_layers(mc_ablation_model)[target_layer_idx]
        mc_cam = MCAblationCAM(mc_ablation_model, target, channel_batch_size=16, k_trials=50)
        return mc_cam.generate(events, class_idx=class_idx).detach()

def generate_sam(events, target_layer_idx, neuron_type):
    with managed_model_copy() as sam_model:
        functional.reset_net(sam_model)
        target = get_lif_layers(sam_model)[target_layer_idx]
        sam = SpikeActivationMap(sam_model, target)
        return sam.generate(events).detach()

def generate_random_cam(events):
    # events: (T, 1, C, H, W) -> need (T, H, W)
    T, _, _, H, W = events.shape
    return torch.rand((T, H, W), device=events.device)

def mask_events(events, cam, keep_percent):
    # cam: (T,H,W)
    cam_agg = cam.mean(dim=0) # (H,W)
    H, W = cam_agg.shape
    num_keep = max(1, int(cam_agg.numel() * keep_percent))
    
    # Use indices to ensure we don't over-select in case of ties (e.g. many zeros)
    # This fixes the issue where an empty/sparse map results in selecting the whole image
    _, indices = torch.topk(cam_agg.view(-1), num_keep)
    
    mask_flat = torch.zeros_like(cam_agg.view(-1))
    mask_flat[indices] = 1.0
    mask2d = mask_flat.view(H, W)
    
    # Expand mask to (1, 1, 1, H, W) to match (T, 1, C, H, W)
    return events * mask2d.view(1, 1, 1, H, W)

def get_scores(net, inputs, class_idx):
    functional.reset_net(net)
    with torch.no_grad():
        out = net(inputs)
    time_sum = out.sum(dim=0)[0]
    conf = F.softmax(time_sum, dim=0)
    return time_sum[class_idx].item(), conf[class_idx].item()

# --- Data Collection ---

# Store results: results[layer_pair_label][method] = list of dicts
results = defaultdict(lambda: defaultdict(list))
spatial_variances = defaultdict(lambda: defaultdict(list))

target_layer_pairs = get_valid_pairs(model, requested_layer_pairs)
sample_count = 0

print(f"Starting evaluation on {len(target_layer_pairs)} layer pairs...")

if max_samples:
    pbar = tqdm(total=max_samples, desc="Samples Collected")
else:
    pbar = tqdm(total=len(test_loader), desc="Batches Processed")

for events, labels in test_loader:
    if max_samples and sample_count >= max_samples:
        break
        
    # Determine neuron type for this sample (LIF or IF)
    is_single_image = (events.dim() == 4)
    current_neuron_type = neuron.IFNode if is_single_image else neuron.LIFNode
    
    if is_single_image:
        events = events.unsqueeze(1).repeat(1, config["N_TIME_BINS"], 1, 1, 1)
    
    events = events.transpose(0, 1).to(device) # (T, B, C, H, W)
    label = labels[0].item()
    
    # Get baseline prediction
    base_act, base_conf = get_scores(model, events, int(label))
    pred_class_idx = int(model(events).sum(dim=0).argmax().item())
    functional.reset_net(model)
    
    if pred_class_idx != int(label):
        if not max_samples:
            pbar.update(1)
        continue

    for (conv_idx, lif_idx) in target_layer_pairs:
        layer_label = f"L{conv_idx}/{lif_idx}" # Label for plots
        
        # Generate CAMs
        cams = {}
        cams['grad'] = generate_grad_cam(events, pred_class_idx, conv_idx, current_neuron_type)
        cams['ablation'] = generate_ablation_cam(events, pred_class_idx, conv_idx, current_neuron_type)
        cams['mc_ablation'] = generate_mc_ablation_cam(events, pred_class_idx, lif_idx, current_neuron_type)
        cams['sam'] = generate_sam(events, lif_idx, current_neuron_type)
        cams['random'] = generate_random_cam(events)
        
        # Compute Variances
        for m in methods:
            spatial_variances[layer_label][m].append(cam_variance_T_HW(cams[m]))

        # Evaluate Masking (Insertion Game: Keep Top %)
        # A good CAM should have LOW drop in confidence when keeping only the top % salient features.
        # Random baseline should have HIGH drop (input starvation).
            masked_input = mask_events(events, cams[m], top_percent)
            act, conf = get_scores(model, masked_input, pred_class_idx)
            
            # Metrics (i) & (ii): Average Drop (lower is better)
            drop_conf = max(0, base_conf - conf)
            drop_act = max(0, base_act - act)
            
            # Metrics (iii) & (iv): Percent Increase (higher is better)
            # Rate at which masked > original
            increase_conf = 1.0 if conf > base_conf else 0.0
            increase_act = 1.0 if act > base_act else 0.0
            
            res = {
                'percent': p,
                'drop_conf': drop_conf,
                'drop_act': drop_act,
                'increase_conf': increase_conf,
                'increase_act': increase_act,
                'conf': conf,
                'act': act
            }
            results[layer_label][m].append(res)
    del cams
    sample_count += 1
    pbar.update(1)
    torch.cuda.empty_cache()

pbar.close()
print(f"Evaluated {sample_count} samples.")

# --- Plotting ---

# 1. Faithfulness (Confidence Drop) vs Masking %
plot_data = []
for layer_label, method_data in results.items():
    for method, data_list in method_data.items():
        for item in data_list:
            plot_data.append({
                'Layer': layer_label,
                'Method': method,
                'Masking %': item['percent'] * 100,
                'Confidence Drop': item['drop_conf'],
                'Activation Drop': item['drop_act'],
                'Confidence Increase Rate': item['increase_conf'],
                'Activation Increase Rate': item['increase_act']
            })
df = pd.DataFrame(plot_data)

# Plot Confidence Drop per Layer
unique_layers = df['Layer'].unique()
for layer in unique_layers:
    plt.figure(figsize=(8, 6))
    sns.barplot(
        data=df[df['Layer'] == layer], 
        x='Method', 
        y='Confidence Drop', 
        hue='Method'
    )
    plt.title(f"Faithfulness (Lower is Better) - {layer} (Top {top_percent*100:.0f}%)")
    plt.ylabel("Drop in Confidence (Base - Masked)")
    plt.xlabel("Method")
    plt.tight_layout()
    plt.show()

# 2. Spatial Variance per Layer
var_data = []
for layer_label, method_data in spatial_variances.items():
    for method, values in method_data.items():
        for v in values:
            var_data.append({
                'Layer': layer_label,
                'Method': method,
                'Variance': v
            })
df_var = pd.DataFrame(var_data)

plt.figure(figsize=(10, 6))
sns.barplot(data=df_var, x='Layer', y='Variance', hue='Method')
plt.title("Spatial Variance of CAMs across Layers")
plt.ylabel("Mean Spatial Variance")
plt.tight_layout()
plt.show()

# 3. Summary Table
print(f"\n--- Summary: Metrics @ {top_percent * 100:.0f}% Kept ---")

print("\n(i) Average Confidence Drop (Lower is better):")
print(df.groupby(['Layer', 'Method'])['Confidence Drop'].mean().unstack())

print("\n(ii) Average Activation Drop (Lower is better):")
print(df.groupby(['Layer', 'Method'])['Activation Drop'].mean().unstack())

print("\n(iii) Confidence Increase Rate (Higher is better):")
print(df.groupby(['Layer', 'Method'])['Confidence Increase Rate'].mean().unstack())

print("\n(iv) Activation Increase Rate (Higher is better):")
print(df.groupby(['Layer', 'Method'])['Activation Increase Rate'].mean().unstack())